# 3.1 经典推荐算法总结与横向对比

> 阅读版与 Web 应用内容一致；实验数值来自本 Notebook 的已执行输出。

## Goal

在完成五个独立实验后，从任务、表示、训练目标、指标、推理方式和工业边界六个角度进行横向比较。本文不重复完整实现，只提供章节地图、结果快照和选型评论。

## Setup

本 Notebook 的默认真实数据是 **GroupLens MovieLens latest-small：经典评分与邻域任务**。`smoke` 档读取仓库内可审计的确定性切片，`full` 档扩大到官方完整文件；两档都不制造交互、曝光、标签或行为序列。切片规则、源地址、哈希与许可记录在 `data/README.md` 及对应数据目录。

**主要资料：** [MovieLens](https://grouplens.org/datasets/movielens/) · [MF](https://datajobs.com/data-science-repo/Recommender-Systems-[Netflix].pdf) · [FM](https://www.csie.ntu.edu.tw/~b97053/paper/Rendle2010FM.pdf) · [GBDT+LR](https://research.facebook.com/publications/practical-lessons-from-predicting-clicks-on-ads-at-facebook/)

In [ ]:
from pathlib import Path
import os, sys, json
import torch
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
ARTIFACT_ROOT = Path(os.environ.get("RECSYS_ARTIFACT_ROOT", PROJECT_ROOT)).expanduser().resolve()
sys.path.insert(0, str(PROJECT_ROOT))
os.environ.setdefault("RECSYS_PROFILE", "full")
PROFILE = os.environ["RECSYS_PROFILE"]
CUDA_AVAILABLE = torch.cuda.is_available()
from recsys_lab.data import (load_movielens, movielens_provenance, load_amazon_2023,
                             amazon_provenance, load_kuairand, kuairand_provenance,
                             load_amazon_2018, amazon_2018_provenance,
                             load_movielens_1m, movielens_1m_provenance,
                             load_mind_amazon_books, mind_amazon_provenance,
                             load_census_income, census_income_provenance)
DATASET_KEY = "movielens"
if DATASET_KEY == "movielens":
    real_ratings, real_movies = load_movielens()
    real_interactions = real_ratings
    REAL_DATASET = movielens_provenance(real_ratings)
elif DATASET_KEY == "amazon-books":
    real_ratings = load_amazon_2018("Books") if PROFILE == "full" else load_amazon_2023()
    real_interactions, real_movies = real_ratings, None
    REAL_DATASET = amazon_2018_provenance(real_ratings) if PROFILE == "full" else amazon_provenance(real_ratings)
elif DATASET_KEY == "mind-amazon-books":
    real_ratings = load_mind_amazon_books() if PROFILE == "full" else load_amazon_2023()
    real_interactions, real_movies = real_ratings, None
    REAL_DATASET = mind_amazon_provenance(real_ratings) if PROFILE == "full" else amazon_provenance(real_ratings)
elif DATASET_KEY == "amazon-electronics":
    real_ratings = load_amazon_2018("Electronics") if PROFILE == "full" else load_amazon_2023()
    real_interactions, real_movies = real_ratings, None
    REAL_DATASET = amazon_2018_provenance(real_ratings) if PROFILE == "full" else amazon_provenance(real_ratings)
elif DATASET_KEY == "movielens-1m":
    real_ratings = load_movielens_1m() if PROFILE == "full" else load_amazon_2023()
    real_interactions, real_movies = real_ratings, None
    REAL_DATASET = movielens_1m_provenance(real_ratings) if PROFILE == "full" else amazon_provenance(real_ratings)
elif DATASET_KEY == "census-income" and PROFILE == "full":
    census_train_x, census_train_y, census_test_x, census_test_y = load_census_income()
    real_interactions, real_movies, real_ratings = census_train_x, None, census_train_x
    REAL_DATASET = census_income_provenance()
elif DATASET_KEY == "amazon-2023":
    real_ratings = load_amazon_2023()
    real_interactions, real_movies = real_ratings, None
    REAL_DATASET = amazon_provenance(real_ratings)
else:
    real_interactions, real_movies = load_kuairand()
    real_ratings = real_interactions
    REAL_DATASET = kuairand_provenance(real_interactions)
print({"profile": PROFILE, "project_root": str(PROJECT_ROOT), "artifact_root": str(ARTIFACT_ROOT), "real_dataset": REAL_DATASET,
       "cuda_available": CUDA_AVAILABLE,
       "cuda_device": torch.cuda.get_device_name(0) if CUDA_AVAILABLE else None})
assert REAL_DATASET["randomly_fabricated_rows"] == 0

## 学习路线

四种算法其实对应三种不同的“信息压缩”方式：

| 方法 | 压缩对象 | 学到的核心信息 | 常见位置 |
|---|---|---|---|
| UserCF / ItemCF | 共现邻域 | 谁和谁相似 | 召回、相关推荐 |
| MF | user–item 矩阵 | 用户与物品的低维隐语义 | 召回、评分预测 |
| FM | 稀疏特征交叉 | 任意两个特征的低秩交互 | CTR 排序 |
| GBDT+LR | 表格特征与树规则 | 非线性分桶、条件组合与概率校准 | CTR 排序 |

建议按顺序运行。前两类使用 user–item 行为；后两类使用“曝光—点击”样本。二者的数据生成机制和评价指标不同，不能把 RMSE、Recall 与 AUC 混为一谈。

## 来源论文与本章解读

本章不是按发表年份罗列名词，而是沿着“表示什么、怎样推理”阅读五条原始工作：

- **Resnick et al. (1994), GroupLens**：早期系统展示了如何依据用户间评分相似度形成邻域预测；关键遗产是“从群体行为借信号”，而不是某个固定相似度公式。
- **Sarwar et al. (2001), Item-based CF**：把邻域移到更稳定、可离线物化的 item 侧，直接影响了后来的相关推荐和倒排召回。
- **Koren, Bell & Volinsky (2009), Matrix Factorization Techniques**：以全局/用户/物品偏置加低秩内积解释评分；其 user/item 向量结构也是双塔召回的最小原型。
- **Rendle (2010), Factorization Machines**：用特征隐向量共享稀疏二阶交叉统计，并用恒等式把计算从 $O(n^2k)$ 降为 $O(nk)$。
- **He et al. (2014), Practical Lessons from Predicting Clicks on Ads at Facebook**：树负责产生非线性叶规则，LR 负责稀疏组合与概率输出，代表经典的两阶段 CTR 工程路线。
- **Mikolov et al. (2013), Efficient Estimation of Word Representations**：用中心词预测上下文高效学词向量；推荐系统把行为序列当句子学物品向量（Item2Vec），是向量召回的起点。

阅读时请区分两组问题：CF/MF/word2vec 面向 user–item 评分或 Top-K 召回；FM/GBDT+LR 面向曝光后的点击概率。不同任务的指标不能直接排名。

## 论文关联关系：后一篇在修补前一篇的哪块短板？

六篇论文不是同一赛道上的名词罗列，而是一条“发现问题 → 换表示 → 暴露新问题”的接力。下表从左到右读：每一行先看它从哪个问题出发，再看它把什么难题交给了下一位。

In [ ]:
import pandas as pd

paper_relationships = pd.DataFrame([
    {"论文": "Resnick 1994 GroupLens", "出发点问题": "netnews 信息过载，编辑过滤跟不上，让读者互评互助",
     "用户/物品表示": "用户评分向量 + 皮尔逊近邻", "训练信号": "1–5 星显式评分",
     "交给下一步的问题": "邻域计算随规模变贵；评分稀疏时找不到邻居"},
    {"论文": "Sarwar 2001 ItemCF", "出发点问题": "用户兴趣多变、用户邻域不稳定且难扩展",
     "用户/物品表示": "物品列向量 + 物品间相似度", "训练信号": "同一批用户在物品上的共现评分",
     "交给下一步的问题": "仍是显式邻域，无法概括潜在口味结构"},
    {"论文": "Koren 2009 MF", "出发点问题": "邻域无法表达潜在结构，Netflix 规模上难以扩展",
     "用户/物品表示": "用户/物品各一个 d 维隐向量，内积打分", "训练信号": "已观察评分上的平方误差",
     "交给下一步的问题": "只认 ID，新实体冷启动；无法直接利用上下文特征"},
    {"论文": "Rendle 2010 FM", "出发点问题": "MF 只认评分矩阵格式；稀疏特征交叉的独立参数学不到",
     "用户/物品表示": "每个特征一个 k 维隐向量，内积共享交叉", "训练信号": "任意监督标签（评分或点击）",
     "交给下一步的问题": "只有二阶交叉，高阶模式需要更深的结构"},
    {"论文": "He 2014 GBDT+LR", "出发点问题": "工业 CTR 同时要非线性规则、校准概率与快速更新",
     "用户/物品表示": "树叶子 one-hot（离散规则）", "训练信号": "曝光—点击日志的 log loss / NE",
     "交给下一步的问题": "两阶段非端到端，特征与树版本靠人工治理"},
    {"论文": "Mikolov 2013 word2vec", "出发点问题": "词（物品）只是编号，需要可计算、可检索的相似度",
     "用户/物品表示": "每个物品一个嵌入向量", "训练信号": "序列窗口内共现（负采样）",
     "交给下一步的问题": "单向量平均多兴趣、序列建模弱 → 3.2 双塔/多兴趣/序列召回"},
])
display(paper_relationships)

## 未来发展：从经典方法走向深度章节

经典方法并没有被“淘汰”，而是各自指出了下一步要解除的约束。下表按四个维度对照 3.1 与后续 3.2/3.3 章的做法；最后一列是仍未回答的问题，不是收益承诺。

In [ ]:
future = pd.DataFrame([
    {"维度": "表示", "3.1 经典做法": "邻域、单隐向量、特征向量",
     "3.2/3.3 深度做法": "双塔 Embedding（DSSM）、多兴趣（MIND）、序列表示（SASRec）",
     "仍待回答": "表示如何随上下文实时变化，同时保持训练/服务一致？"},
    {"维度": "特征交互", "3.1 经典做法": "内积、FM 二阶交叉、树规则离散组合",
     "3.2/3.3 深度做法": "DeepFM 共享 embedding 端到端学高阶交叉；DIN/DIEN 候选感知兴趣激活与演化",
     "仍待回答": "高阶交互的可解释性、计算成本与过拟合控制？"},
    {"维度": "训练信号", "3.1 经典做法": "显式评分平方误差、点击 log loss、序列共现负采样",
     "3.2/3.3 深度做法": "in-batch softmax 召回目标、辅助损失、next-item 序列目标",
     "仍待回答": "曝光/位置偏差纠正、多目标（点击/转化/时长）如何对齐？"},
    {"维度": "服务", "3.1 经典做法": "离线近邻表、物品向量 ANN、两阶段树+LR",
     "3.2/3.3 深度做法": "用户/物品塔分离 + ANN 召回，统一排序模型在线打分",
     "仍待回答": "索引新鲜度、级联一致性与 P99 成本如何工程化？"},
])
display(future)

## Steps

## 1. 五个独立实验

| Notebook | 核心问题 | 主要指标 |
|---|---|---|
| 3.1.1 协同过滤 | 谁与谁相似，如何用邻域产生候选？ | HR@K、Recall@K、Coverage |
| 3.1.2 矩阵分解（BiasMF） | 如何把稀疏矩阵压缩成用户/物品隐向量？ | RMSE、HR@K |
| 3.1.3 FM | 如何在线性复杂度内学习稀疏二阶交互？ | AUC、LogLoss |
| 3.1.4 GBDT+LR | 如何用树叶规则生成特征，再由 LR 校准概率？ | AUC、LogLoss |
| 3.1.5 word2vec / Item2Vec | 如何把行为序列学成物品向量做召回？ | Recall@K、Coverage |

这五篇均可独立从头执行，不依赖其他 Notebook 的内存状态。

## 2. 结果快照

以下数值由五个算法 Notebook 在执行末尾写入 `results/chapter_3_1/*.json`，本总结只读取产物，不手填实验值。CF/MF/word2vec 与 FM/GBDT+LR 属于不同任务，表格用于确认代码路径和理解指标，不能按数值大小直接排名。

In [ ]:
import json
import pandas as pd

result_dir = ARTIFACT_ROOT / "results" / "chapter_3_1"
result_files = sorted(result_dir.glob("*.json"))
assert len(result_files) == 5, f"需要先执行五个算法 Notebook；当前产物：{[p.name for p in result_files]}"
records = []
for path in result_files:
    payload = json.loads(path.read_text(encoding="utf-8"))
    records.extend(payload["records"])
comparison = pd.DataFrame(records)
display(comparison.round(4))
print("指标来源：", [p.name for p in result_files])

## 3. 与原论文的可比性审计

本章算法来自不同任务，原论文也没有一个统一 benchmark。下表的目的不是制造“复现率”，而是明确当前教学实验与论文实验之间的边界。

| 算法 | 当前教程产物 | 原论文代表性结果/规模 | 审计结论 |
|---|---|---|---|
| UserCF / ItemCF | MovieLens latest-small 子集，HR@10=0.125 | GroupLens 与 ItemCF 论文采用更早的数据与 MAE/预测质量等口径 | 实现验证，不是原论文复现；指标定义不同，不能相减。 |
| BiasMF | RMSE=1.1905 | Koren 等报告 Netflix 系统 RMSE=0.9514、大奖目标 0.8563；Netflix 含约 1 亿评分 | 当前小样本结果明显更弱，但数据、切分、维数与正则均未对齐。 |
| FM | AUC=0.5964，LogLoss=2.6061 | FM 原论文重点验证稀疏因子化的通用性，包含 Netflix 1 亿样本等回归/排序任务，并未给出本教程 CTR 口径 | AUC 略高于随机，但 LogLoss 显示严重过度自信；只能作为公式实现练习。 |
| GBDT+LR | AUC=0.6376，LogLoss=0.7414 | Facebook 论文使用工业广告日志与 Normalized Entropy、校准等内部口径 | 当前仅验证“树叶编码 → LR”链路，不能宣称复现工业收益。 |
| word2vec / Item2Vec | Recall@5=0.05，Coverage≈0.18 | Mikolov 在文本词向量任务上评测句法/语义类比；Item2Vec 论文用协同过滤数据 | 小序列 smoke 仅验证 Skip-gram 链路与向量近邻，不能对标论文词向量质量。 |

因此 3.1 的可信结论是代码链路、数学结构和同一教程内的基线差异；不能把这些 smoke 数字当成公开 benchmark。

## 4. 横向评论

| 维度 | CF | MF | FM | GBDT+LR | word2vec |
|---|---|---|---|---|---|
| 输入 | user–item 行为 | user–item 评分/行为 | 任意稀疏与连续特征 | 表格特征 | 行为序列 |
| 表示 | 邻域/相似度 | 低维 embedding | 特征 embedding | 树叶规则 | 物品 embedding |
| 交互能力 | 共现传播 | user-item 内积 | 全部二阶交互 | 非线性条件组合 | 序列共现 |
| 冷启动 | 弱 | 弱 | 可加入内容特征 | 可加入内容特征 | 弱（新物品无向量） |
| 序列能力 | 无 | 无 | 无 | 无 | 局部次序 |
| 典型位置 | 召回/相关推荐 | 召回/评分 | CTR 排序 | CTR 排序 | 向量召回 |
| 主要风险 | 稀疏、热门偏差 | 未观察样本、内积限制 | 仅二阶 | 两阶段失配、规则漂移 | 短序列、热门偏置 |

技术演进不是简单替换关系：ItemCF 常作为覆盖与兜底通道长期保留；MF 是双塔向量召回的最小原型；FM 和 GBDT+LR 则代表排序阶段两条经典特征交叉路线；word2vec / Item2Vec 把序列嵌入做成可 ANN 的召回向量，衔接后续 DSSM、SASRec。

## Checks

- 比较 Top-K 模型时同时看命中、覆盖和新颖度。
- 比较 CTR 模型时至少同时看 AUC、LogLoss 与校准。
- 只在相同数据、相同时间切分、相同负样本下横向比较。
- smoke 指标用于代码与理解验证，不当作公开 benchmark。

In [ ]:
assert len(comparison) == 6
assert set(comparison.task) == {'Top-K', '评分预测', 'CTR'}
assert comparison.source_notebook.nunique() == 5
print('PASS：章节总结从五个独立实验聚合了全部经典算法与三类任务。')

## Next Steps

按顺序阅读五篇算法 Notebook。若目标是构建系统基线，建议先落地 ItemCF 与 GBDT+LR/FＭ，再根据目录规模和特征类型引入 MF/双塔与 DeepFM。